# AGAR-RL: Autonomous Multi-Agent Deep Reinforcement Learning Pipeline

Ce notebook permet d'exécuter l'entraînement distribué par Deep Reinforcement Learning (PPO & Self-Play Multi-Modèles) directement sur **Google Colab** (GPU T4/L4/A100) avec **sauvegarde automatique et reprise continue sur Google Drive**.

## 0. Connexion & Test de Sauvegarde sur Google Drive (Anti-Perte de Données)
Exécutez cette cellule en premier pour monter votre Google Drive. Tous vos checkpoints de modèles y seront sauvegardés en temps réel. Si la session Colab expire ou se coupe, vos modèles ne seront **jamais perdus** et pourront reprendre automatiquement !

In [ ]:
# 1. Montage de Google Drive
from google.colab import drive
import os, time

drive.mount('/content/drive')

# 2. Création du dossier de sauvegarde dédié sur votre Drive
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/agario_rl_backup'
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

# 3. Test immédiat d'écriture et de lecture
test_file = os.path.join(DRIVE_BACKUP_DIR, 'test_connection.txt')
with open(test_file, 'w', encoding='utf-8') as f:
    f.write(f'Connexion Google Drive OK - {time.ctime()}\n')

with open(test_file, 'r', encoding='utf-8') as f:
    status = f.read().strip()

print(f'✅ {status}')
print(f'📁 Dossier de sauvegarde actif : {DRIVE_BACKUP_DIR}')


## 1. Détection de l'Environnement et Installation des Dépendances

In [ ]:
import os, sys

# 1. Récupération des dernières modifications ou clonage
if os.path.exists('.git'):
    print('🔄 Récupération des dernières mises à jour du repo...')
    !git pull origin main
elif os.path.exists('agario/.git'):
    print('🔄 Déplacement dans agario et mise à jour...')
    %cd agario
    !git pull origin main
else:
    print('🌐 Environnement distant Colab détecté. Clonage du repo...')
    !git clone https://github.com/Albin0903/agario.git
    %cd agario

# 2. Configuration du PYTHONPATH et installation des dépendances Farama Gymnasium
os.environ['PYTHONPATH'] = f"{os.getcwd()}:{os.environ.get('PYTHONPATH', '')}"
!pip uninstall -y -q gym 2>/dev/null || true
!pip install -q -r requirements.txt tensorboard
!apt-get install -qq -y ffmpeg

# 3. Vérification GPU CUDA
import torch
print(f'CUDA disponible : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'🚀 GPU actif : {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ ATTENTION : Vous êtes sur processeur (CPU) ! Cliquez sur Exécution > Modifier le type d\'exécution > et sélectionnez GPU T4 ou GPU L4.')


## 2. Validation de la Suite de Tests (27 Tests)

In [ ]:
!python -m pytest -v

## 3. Monitoring TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/tensorboard

## 4. Démarrer l'Entraînement Multi-Agent (Sauvegarde Google Drive & Auto-Resume)

In [ ]:
# 🚀 Entraînement long et intensif (5 000 000 timesteps ≈ 2h30 sur GPU T4 / ~1h15 sur GPU L4)
# --backup-dir /content/drive/MyDrive/agario_rl_backup : copie chaque checkpoint sur Drive en temps réel
# --resume auto : reprend automatiquement là où vous vous étiez arrêté si un checkpoint existe sur Drive !
!python src/training/train_colab.py \
    --n-envs 16 \
    --total-timesteps 5000000 \
    --pool-interval 200000 \
    --backup-dir /content/drive/MyDrive/agario_rl_backup \
    --resume auto \
    --device auto


## 5. Visualiser & Enregistrer les Parties en Vidéo HD (Replay avec Vecteurs de Décision)

In [ ]:
# 1. Recherche automatique du dernier checkpoint disponible (Drive ou local)
import os, glob
from IPython.display import HTML, display
from base64 import b64encode

drive_ckpts = sorted(glob.glob('/content/drive/MyDrive/agario_rl_backup/*.zip'))
local_ckpts = sorted(glob.glob('checkpoints/self_play_pool/*.zip'))
all_ckpts = drive_ckpts or local_ckpts
latest_ckpt = all_ckpts[-1] if all_ckpts else 'checkpoints/ppo/ppo_latest.zip'
print(f'🎬 Modèle sélectionné pour le replay : {latest_ckpt}')

# 2. Enregistrement HD (2400 steps = 80 secondes de vidéo @ 30 FPS)
os.makedirs('recordings', exist_ok=True)
!python src/inference/record_match.py \
    --model {latest_ckpt} \
    --output recordings/eval_match.mp4 \
    --steps 2400

# 3. Sauvegarde de la vidéo sur Google Drive
if os.path.exists('recordings/eval_match.mp4') and os.path.exists('/content/drive/MyDrive/agario_rl_backup'):
    !cp recordings/eval_match.mp4 /content/drive/MyDrive/agario_rl_backup/eval_match.mp4
    print('📁 Vidéo sauvegardée sur Google Drive dans agario_rl_backup/eval_match.mp4')

# 4. Visualisation directe dans le Notebook
video_path = 'recordings/eval_match.mp4'
if os.path.exists(video_path):
    mp4_bytes = open(video_path, 'rb').read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4_bytes).decode()
    display(HTML(f'''
    <video width="800" height="450" controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    '''))
    print(f'Taille de la vidéo : {os.path.getsize(video_path) / 1_000_000:.1f} Mo')
else:
    print('Erreur : vidéo non trouvée.')


## 6. Exporter la Politique vers ONNX (< 0.02 ms de latence CPU)

In [ ]:
# 1. Export ONNX
!python src/inference/export_onnx.py \
    --model checkpoints/ppo/ppo_final.zip \
    --output models/model.onnx

# 2. Sauvegarde du modèle ONNX sur Google Drive
if os.path.exists('models/model.onnx') and os.path.exists('/content/drive/MyDrive/agario_rl_backup'):
    !cp models/model.onnx /content/drive/MyDrive/agario_rl_backup/model.onnx
    print('📁 Modèle ONNX sauvegardé sur Google Drive dans agario_rl_backup/model.onnx')
